In [2]:
import duckdb
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
import pyproj


In [19]:
rasterio_proj_dir = r"C:\Users\Lilly\anaconda3\envs\msc\Lib\site-packages\rasterio\proj_data"
print(os.path.exists(rasterio_proj_dir))  # should be True


True


In [20]:
os.environ["PROJ_LIB"] = rasterio_proj_dir
os.environ["PROJ_DATA"] = rasterio_proj_dir

import rasterio  # re-import after setting env vars, in case it wasn't imported yet

In [7]:
db = duckdb.read_parquet(r"data/input/piebro/changeset_data/year=*/month=*/*.parquet", hive_partitioning=1) 

### Per grid

In [8]:
#for the total of the 2020-2025 timeframe

df_grid_ai = duckdb.sql("""
    SELECT
        COUNT(DISTINCT user_name) AS Contributors_AI,
        CAST(SUM(edit_count) AS FLOAT) AS Edits_AI,
        CAST(COUNT(*) AS FLOAT) AS Changesets_AI,
        mid_pos_x,
        mid_pos_y
    FROM db
    WHERE year >= 2020 AND year <2026 AND mid_pos_x IS NOT NULL AND mid_pos_y IS NOT NULL
      AND (created_by = 'Rapid'
         OR array_to_string(source, ',') ILIKE '%microsoft/BuildingFootprints%'
         OR array_to_string(source, ',') ILIKE '%mapwithai%'
         OR array_to_string(source, ',') ILIKE '%esri/Google_Africa_Buildings%'
         OR array_to_string(source, ',') ILIKE '%esri/Google_Open_Buildings%'
         OR array_to_string(hashtags, ',') ILIKE '%mapwithai%'
      )
    GROUP BY mid_pos_x, mid_pos_y
""").df()

In [9]:
df_grid_ai

,Contributors_AI,Edits_AI,Changesets_AI,mid_pos_x,mid_pos_y
0,6,590.0,8.0,212,137
1,74,16243.0,450.0,175,141
2,8,421.0,20.0,173,142
3,30,36209.0,243.0,72,127
4,44,193919.0,765.0,213,92
...,...,...,...,...,...
9736,1,290.0,1.0,149,128
9737,1,5.0,1.0,113,49
9738,1,306.0,1.0,207,124
9739,1,2.0,2.0,276,146


In [10]:
#for the total of the 2020-2025 timeframe

df_grid_total = duckdb.sql("""
    SELECT
        COUNT(DISTINCT user_name) AS Contributors,
        CAST(SUM(edit_count) AS FLOAT) AS Edits,
        CAST(COUNT(*) AS FLOAT) AS Changesets,
        mid_pos_x,
        mid_pos_y
    FROM db
    WHERE year >= 2020 AND year <2026 AND mid_pos_x IS NOT NULL AND mid_pos_y IS NOT NULL
    GROUP BY mid_pos_x, mid_pos_y
""").df()

In [11]:
df_grid_total =df_grid_total.merge(df_grid_ai, on =("mid_pos_x", "mid_pos_y"), how="left")

In [12]:
df_grid_total

,Contributors,Edits,Changesets,mid_pos_x,mid_pos_y,Contributors_AI,Edits_AI,Changesets_AI
0,581,884360.0,7102.0,211,65,5.0,237.0,7.0
1,1401,3298250.0,35020.0,104,95,40.0,156708.0,586.0
2,99,40560.0,456.0,94,137,1.0,3.0,1.0
3,1559,1364545.0,37112.0,241,147,7.0,125.0,16.0
4,3472,4784421.0,113911.0,197,138,16.0,39938.0,237.0
...,...,...,...,...,...,...,...,...
24757,1,765.0,4.0,152,79,NaN,NaN,NaN
24758,1,4.0,1.0,91,115,NaN,NaN,NaN
24759,1,5.0,1.0,42,6,NaN,NaN,NaN
24760,1,1.0,1.0,335,82,NaN,NaN,NaN


In [13]:
df_grid_total["ai_edit_share"] = (df_grid_total["Edits_AI"] /  df_grid_total["Edits"] )*100

In [14]:
df_grid_total.sort_values(by="ai_edit_share", ascending=False)

,Contributors,Edits,Changesets,mid_pos_x,mid_pos_y,Contributors_AI,Edits_AI,Changesets_AI,ai_edit_share
2925,1,494.0,1.0,269,101,1.0,494.0,1.0,100.0
2913,1,57.0,1.0,55,136,1.0,57.0,1.0,100.0
2888,1,214.0,1.0,76,107,1.0,214.0,1.0,100.0
19923,1,509.0,1.0,342,51,1.0,509.0,1.0,100.0
4451,1,16.0,1.0,99,63,1.0,16.0,1.0,100.0
...,...,...,...,...,...,...,...,...,...
24757,1,765.0,4.0,152,79,NaN,NaN,NaN,NaN
24758,1,4.0,1.0,91,115,NaN,NaN,NaN,NaN
24759,1,5.0,1.0,42,6,NaN,NaN,NaN,NaN
24760,1,1.0,1.0,335,82,NaN,NaN,NaN,NaN


In [15]:
pivottotal = df_grid_total.pivot_table(index="mid_pos_y", columns="mid_pos_x", values="ai_edit_share", aggfunc="sum")


In [16]:
print("non-null cells:", pivottotal.notna().sum().sum())
print("non-null AND non-zero cells:", (pivottotal.notna() & (pivottotal != 0)).sum().sum())


non-null cells: 24762
non-null AND non-zero cells: 9741


In [21]:
pivottotal = df_grid_total.pivot_table(
    index="mid_pos_y", columns="mid_pos_x", values="ai_edit_share", aggfunc="sum"
)
pivottotal.index = pivottotal.index.astype(np.int64)
pivottotal.columns = pivottotal.columns.astype(np.int64)

res = 1.0
pivottotal = pivottotal.reindex(
    index=np.arange(0, 181, dtype=np.int64),
    columns=np.arange(0, 361, dtype=np.int64),
)
pivottotal = pivottotal.sort_index(ascending=False)

nodata_val = -9999.0
raw = pivottotal.values.astype("float64")
data = np.where(np.isnan(raw), nodata_val, raw).astype("float32")

valid_mask = data != np.float32(nodata_val)
print("valid pixels:", valid_mask.sum(), "sum:", data[valid_mask].sum(), "max:", data[valid_mask].max())
assert valid_mask.sum() > 0

lon_min = -180 - res / 2
lat_max = 90 + res / 2
transform = from_origin(lon_min, lat_max, res, res)

valid pixels: 24762 sum: 80121.66 max: 100.0


In [22]:
with rasterio.open(
    "edit_share_final3.tif", "w",
    driver="GTiff",
    height=data.shape[0], width=data.shape[1],
    count=1, dtype="float32",
    crs="EPSG:4326",
    transform=transform,
    nodata=nodata_val,
) as dst:
    dst.write(data, 1)

In [23]:
with rasterio.open("edit_share_final2.tif") as src:
    arr = src.read(1)
    mask = arr != np.float32(nodata_val)
    print("FILE CHECK — valid pixels:", mask.sum(), "sum:", arr[mask].sum() if mask.sum() else "N/A")

FILE CHECK — valid pixels: 24762 sum: 80121.66
